In [1]:
import pandas as pd
import random

#create a class for the stars
class Star:
    """represents the star with its properties
    name: name of the star
    constellation: constellation of the star
    magnitude: magnitude of the star
    distance: distance of the star
    spectral: spectral type of the star
    """
    def __init__(self, name, constellation, magnitude, distance, spectral):
        self.name = name
        self.constellation = constellation
        self.magnitude = magnitude
        self.distance = distance
        self.spectral = spectral
#first hint: give constellation and distance
    def basic_hint(self):
        """provides the first hint, returns constellation and distance"""
        return f"Constellation: {self.constellation}\nDistance: {self.distance:.2f} ly"
#second hint, add magnitude and spectral type
    def full_hint(self):
        """returns second hint, returns constellation, magnitude, distance, spectral type"""
        return (f"Constellation: {self.constellation}\n"
                f"Distance: {self.distance:.2f} ly\n"
                f"Magnitude: {self.magnitude:.2f}\n"
                f"Spectral type: {self.spectral}")
#open the star data
class StarDatabase:
    """loads random stars from database
    df: containing the star data
    used_indices: set of indices already used"""
    def __init__(self,df):
        self.df = pd.read_csv("hygdata_v42.csv.gz")
        # Filter stars with proper names
        self.df = self.df[self.df['proper'].notna()]
        self.used_indices = set()  #tracking stars

    def get_random_star(self):
        """returns a random star, with avoiding repeats, gives distance in ly"""
        # Avoid repeats
        remaining = set(self.df.index) - self.used_indices
        if not remaining:
            self.used_indices.clear()
            remaining = set(self.df.index)
        idx = random.choice(list(remaining))
        self.used_indices.add(idx)
        row = self.df.loc[idx]
        #convert distance into ly
        distance_ly = row['dist'] * 3.262 if 'dist' in row else 0
        return Star(
            name=row['proper'],
            constellation=row['con'] if 'con' in row and row['con'] else "Unknown",
            magnitude=row['mag'] if 'mag' in row else 0,
            distance=distance_ly,
            spectral=row['spect'] if 'spect' in row and row['spect'] else "Unknown"
        )
#define class for players
class Player:
    """represents a player, with attributes name, score, lives"""
    def __init__(self, name):
        self.name = name   #players name
        self.score = 0
        self.lives = 5     #number of lives
#add points
    def add_score(self, points):
        """adds points to score"""
        self.score += points
#lose a life
    def lose_life(self):
        """subtracts a life"""
        self.lives -= 1
#class for game
class StarExplorer:
    """calss for the game, with attributes rounds and database"""
    def __init__(self, db, rounds=5):
        self.db = db
        self.rounds = rounds
#main game loop
    def start(self):
        """starts game with player input, hints, scoring and lives"""
        #prints instructions
        print("Welcome to Star Explorer Challenge!")
        print("The goal of this game is to guess the correct star, based on the hints given.")
        print("If you guess wrong the first time, you get more hints. You have 5 lives total. Good luck!")
        pname = input("Enter your name: ").strip()  #input your name
        player = Player(pname)

        for i in range(1, self.rounds + 1): #loops the rounds
            print(f"\n--- Round {i} ---")  #prints the rounds
            star = self.db.get_random_star()  #gives a random star
            tries = 2

            while tries > 0:
                if tries == 2:
                    print(star.basic_hint())  #print first hint
                else:
                    print("Extra hint:")
                    print(star.full_hint())   #print advanced hint

                guess = input("Guess the star's name: ").strip()  #input guess
                if guess.lower() == star.name.lower():
                    points = 10 if tries == 2 else 5              #for basic hint, 10 points, for advanced hint 5 points
                    print(f"Correct! You earn {points} points.")
                    player.add_score(points)
                    break
                else:
                    tries -= 1
                    player.lose_life()  #lose life
                    if tries > 0:
                        print(f"Incorrect! Try again. Lives left: {player.lives}") #for 1 hint
                    else:
                        print(f"Incorrect! The correct answer was: {star.name}")   #if not guessed
                        print(f"Lives left: {player.lives}")
                if player.lives == 0:  #if no lives left
                    print("\nGame Over! You lost all your lives.")
                    print(f"Final Score: {player.score}")
                    return

        print("Congratulations! You explored all stars.")
        print(f"Final Score: {player.score}")


if __name__ == "__main__":   #loads database, create game objects and starts the game
    db = StarDatabase("hygdata_v42.csv.gz")
    game = StarExplorer(db, rounds=5)
    game.start()

Welcome to Star Explorer Challenge!
The goal of this game is to guess the correct star, based on the hints given.
If you guess wrong the first time, you get more hints. You have 5 lives total. Good luck!
Enter your name: Isabelle

--- Round 1 ---
Constellation: Ser
Distance: 154.67 ly
Guess the star's name: Sirius
Incorrect! Try again. Lives left: 4
Extra hint:
Constellation: Ser
Distance: 154.67 ly
Magnitude: 4.62
Spectral type: A5V
Guess the star's name: Sirius
Incorrect! The correct answer was: Alya
Lives left: 3

--- Round 2 ---
Constellation: Boo
Distance: 40.80 ly
Guess the star's name: Alya
Incorrect! Try again. Lives left: 2
Extra hint:
Constellation: Boo
Distance: 40.80 ly
Magnitude: 4.83
Spectral type: G2V + G2V
Guess the star's name: Alya
Incorrect! The correct answer was: Quadrans
Lives left: 1

--- Round 3 ---
Constellation: Dra
Distance: 97.43 ly
Guess the star's name: ALya\
Incorrect! Try again. Lives left: 0

Game Over! You lost all your lives.
Final Score: 0
